In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load dataset
df = pd.read_csv("WELFake_Dataset.csv")

# Drop first column (ID column)
df = df.iloc[:, 1:]

# Drop missing values
df = df.dropna()

# Combine title and text
df["content"] = df["title"] + " " + df["text"]

X = df["content"]
y = df["label"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF vectorization
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.7,
    min_df=2,
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train Linear SVM
# LinearSVC is fast but has no predict_proba -> calibrate to get probabilities
base_svm = LinearSVC(class_weight="balanced", random_state=42)
svm = CalibratedClassifierCV(base_svm, method="sigmoid", cv=3)  # cv=3 keeps it fast

svm.fit(X_train_tfidf, y_train)

# Evaluate model on test set
y_pred = svm.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Evaluate on training set
y_train_pred = svm.predict(X_train_tfidf)

print("\nTRAINING SET PERFORMANCE")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print(classification_report(y_train, y_train_pred))

print("\nTEST SET PERFORMANCE")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

print("\nTRAIN confusion matrix:")
print(confusion_matrix(y_train, y_train_pred))

print("\nTEST confusion matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
def test_article(title, text, model, vectorizer, crisis=False):
    content = title + " " + text
    content_tfidf = vectorizer.transform([content])

    probs = model.predict_proba(content_tfidf)[0]
    class_to_prob = dict(zip(model.classes_, probs))

    prob_fake = class_to_prob.get(0, None)
    prob_real = class_to_prob.get(1, None)
    if prob_fake is None or prob_real is None:
        raise ValueError(f"Expected classes [0, 1], but got {model.classes_}")

    threshold = 0.8 if crisis else 0.5
    classification = "Most likely real" if prob_real >= threshold else "Most likely fake"
    return classification, prob_real, prob_fake


title = "Scientists Confirm Drinking Coffee Makes People Live to 150 Years Old"
text = (
    "A group of scientists reportedly discovered that drinking three cups of coffee a day can extend "
    "human life to over 150 years. The study, which has not yet been published or reviewed, claims "
    "participants showed near-immortal cellular behavior."
)

result, prob_real, prob_fake = test_article(
    title=title,
    text=text,
    model=svm,           # <-- Linear SVM (calibrated)
    vectorizer=vectorizer,
    crisis=True
)

print("Classification:", result)
print("Probability real:", round(prob_real, 3))
print("Probability fake:", round(prob_fake, 3))
